# NeuroLIF-SKY130 — PVT and Deterministic Variation Analysis

This notebook summarizes the final robustness validation used to
freeze the transistor-level NeuroLIF-SKY130 design before physical
implementation.

Two validation sets are included:

1. **27-point PVT sweep**
   - FF, TT and SS process corners
   - 1.62 V, 1.80 V and 1.98 V
   - −40 °C, 27 °C and 125 °C

2. **13 deterministic parameter-variation conditions**
   - membrane capacitance
   - synaptic reference current
   - leak bias

The deterministic parameter sweep is **not Monte Carlo mismatch
analysis**.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root():
    start = Path.cwd().resolve()

    candidates = [
        start,
        *start.parents,
    ]

    for candidate in candidates:

        if (
            (candidate / "results").exists()
            and
            (candidate / "layout").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate NeuroLIF-SKY130 repository root."
    )


ROOT = find_repo_root()

print("Repository root:", ROOT)


In [ ]:
PVT_FILE = (
    ROOT
    / "results"
    / "pvt"
    / "pvt_27point_summary.csv"
)

VAR_FILE = (
    ROOT
    / "results"
    / "variation"
    / "variation_13case_summary.csv"
)


pvt_df = pd.read_csv(
    PVT_FILE
)

variation_df = pd.read_csv(
    VAR_FILE
)


print(
    "PVT rows:",
    len(pvt_df)
)

print(
    "Variation rows:",
    len(variation_df)
)


## PVT validation


In [ ]:
pvt_pass = (
    pvt_df["point_pass"]
    .astype(str)
    .str.lower()
    .eq("true")
)


print(
    "PVT points passed:",
    int(pvt_pass.sum()),
    "/",
    len(pvt_df),
)


assert len(pvt_df) == 27
assert pvt_pass.all()


pvt_df


In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 4.8)
)


for weight in [
    "01",
    "10",
    "11",
]:

    ax.plot(
        range(len(pvt_df)),
        pvt_df[
            f"spikes_{weight}"
        ],
        marker="o",
        label=f"Weight {weight}",
    )


ax.set_xlabel(
    "PVT point index"
)

ax.set_ylabel(
    "Spike count"
)

ax.set_title(
    "Firing response across 27 PVT points"
)

ax.legend()

ax.grid(
    alpha=0.25,
)

plt.tight_layout()
plt.show()


Some PVT points may suppress lower-weight firing while the final
functional criteria still pass. The robustness gate checks the
required circuit behavior rather than demanding an identical
spike count at every operating point.


## Deterministic parameter variation


In [ ]:
variation_pass = (
    variation_df[
        "condition_pass"
    ]
    .astype(str)
    .str.lower()
    .eq("true")
)


print(
    "Variation conditions passed:",
    int(
        variation_pass.sum()
    ),
    "/",
    len(
        variation_df
    ),
)


assert (
    len(
        variation_df
    )
    == 13
)

assert variation_pass.all()


variation_df


In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 4.8)
)


ax.plot(
    variation_df["condition"],
    variation_df["spikes_11"],
    marker="o",
)


ax.set_xlabel(
    "Deterministic variation condition"
)

ax.set_ylabel(
    "Weight-11 spike count"
)

ax.set_title(
    "Weight-11 response under deterministic variation"
)

ax.tick_params(
    axis="x",
    rotation=60,
)

ax.grid(
    alpha=0.25,
)

plt.tight_layout()
plt.show()


In [ ]:
FULL_PVT_PASS = bool(
    len(pvt_df) == 27
    and
    pvt_pass.all()
)


DETERMINISTIC_VARIATION_PASS = bool(
    len(variation_df) == 13
    and
    variation_pass.all()
)


FINAL_ROBUSTNESS_SUMMARY_PASS = bool(

    FULL_PVT_PASS

    and

    DETERMINISTIC_VARIATION_PASS
)


print(
    "FULL_PVT_PASS =",
    FULL_PVT_PASS,
)

print(
    "DETERMINISTIC_VARIATION_PASS =",
    DETERMINISTIC_VARIATION_PASS,
)

print(
    "FINAL_ROBUSTNESS_SUMMARY_PASS =",
    FINAL_ROBUSTNESS_SUMMARY_PASS,
)


assert FINAL_ROBUSTNESS_SUMMARY_PASS


## Final robustness status

The frozen public data records:

- **27 / 27 PVT points passed**
- **13 / 13 deterministic variation conditions passed**

These results supported the pre-layout circuit freeze.

The deterministic sweep should not be described as process Monte
Carlo or transistor mismatch analysis.
